# Notebook 05 — Stage H Coupled Pipeline Design (Synthetic + Registry)

Purpose:
- Move to next phase with a **design-first notebook**.
- Define how synthetic and future registry data are coupled into one reusable pipeline.
- Define stage-wise model progression without locking into one branch too early.

Important:
- This notebook is intentionally design-focused.
- No hard proof-by-code required in this phase.
- Implementation cells can be added incrementally once coupling decisions are approved.

## 1) Coupling Principle

We couple **at the canonical contract layer**, not at raw schema layer.

Architecture:
1. Source-specific adapters
   - `synthetic_adapter`
   - `registry_adapter`
2. Shared canonical contract
3. Shared feature/label pipeline
4. Shared evaluation and interpretation pipeline

Meaning:
- Synthetic and registry can differ in raw shape/quality.
- But once adapted, they become interchangeable for downstream modeling/evaluation.

## 2) Canonical Contract (Design Draft)

Core timeline fields:
- `patient_id`
- `day` (or standardized time index)
- `cycle_id`
- `months_since_cycle_start`

Core signal fields:
- `instability_signal`
- `hazard_proxy`
- `response_state`
- missingness indicators (`*_is_missing`)

Outcome fields:
- `escalation_event`
- `admission_event` (optional depending on stage)

Governance fields:
- `data_source` (`synthetic`/`registry`)
- `cohort_version`
- `adapter_version`
- `split_set` (frozen train/val/test assignment)

## 3) Adapter Responsibilities (No Modeling Inside Adapters)

`synthetic_adapter` does:
- Map Stage F outputs into canonical contract names
- Apply dtype normalization
- Preserve all rows including extremes and missingness

`registry_adapter` does:
- Map real registry columns/codes into canonical contract names
- Standardize time granularity
- Preserve uncertainty/missingness as first-class signals

Adapters must NOT:
- Tune model features using outcome metrics
- Drop difficult subgroups silently
- Leak future information

## 4) Stage-wise Model Progression (Planned)

### H0 — Governance Baseline
- Frozen split policy
- Leakage checklist
- Data contract validation

### H1 — Baseline Explainable Models
- Elastic-net logistic baseline
- Calibration + uncertainty outputs
- Error table by subgroup

### H2 — Strong Tabular Models
- RandomForest, XGBoost/LightGBM
- Compare against H1 using same split and metrics

### H3 — Messy Data Story Models
- Keep missingness and extremes explicitly in feature space
- Add interaction features only if story-consistent

### H4 — Cross-Source Generalization
- Train synthetic -> evaluate registry
- Train registry -> evaluate synthetic
- Report drift, calibration shift, and story consistency

## 5) Evaluation Stack (Same for Every Stage)

Primary metrics:
- AUROC
- PR-AUC
- Brier
- ECE

Stability metrics:
- Bootstrap confidence intervals
- Subgroup metric spread

Story metrics:
- Extreme vs non-extreme event-risk ratio
- Missingness-as-signal contribution
- Feature permutation importance rank stability

## 6) Decision Gates to Prevent Silent Overfitting

A stage is promoted only if:
1. Performance improves or is equivalent with lower complexity,
2. Calibration does not degrade materially,
3. Subgroup failure does not worsen,
4. Story metrics remain clinically coherent.

If not, stage is marked exploratory and not promoted.

## 7) Next Implementation Entry Point

When you’re ready to implement, start in this notebook with:
- Adapter interface skeleton (`load_source`, `normalize_contract`)
- Shared metric runner
- H1 baseline model block

And keep Notebook 04 as the Stage G messy-story experimentation reference.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook05_stage_h'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook05_stage_h'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook05_stage_h'
META_DIR = PROJECT_ROOT / 'Data' / 'metadata'
for d in [TABLE_DIR, FIG_DIR, REPORT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Stage H implementation workspace ready')

Stage H implementation workspace ready


In [2]:
panel_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_f' / 'phase_f_closed_loop_panel.parquet'
cols = [
    'patient_id', 'day', 'I_stage_f_base', 'hazard_prob_stage_f', 'stage_f_escalation_event',
    'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active'
 ]
df = pd.read_parquet(panel_path, columns=cols).copy()

for c in ['I_stage_f_base', 'hazard_prob_stage_f', 'months_since_cycle_start']:
    df[f'{c}__is_missing'] = df[c].isna().astype(np.int8)
df['is_nonresponse'] = df['response_state_active'].eq('nonresponse').astype(np.int8)
df['is_partial'] = df['response_state_active'].eq('partial_response').astype(np.int8)
df['is_stabilized'] = df['response_state_active'].eq('stabilized').astype(np.int8)
df['I_stage_f_base_x_cycle'] = df['I_stage_f_base'].fillna(df['I_stage_f_base'].median()) * df['cycle_id_stage_f']

feature_set = [
    'cycle_id_stage_f', 'months_since_cycle_start', 'I_stage_f_base', 'hazard_prob_stage_f',
    'is_nonresponse', 'is_partial', 'is_stabilized',
    'I_stage_f_base__is_missing', 'hazard_prob_stage_f__is_missing', 'months_since_cycle_start__is_missing',
    'I_stage_f_base_x_cycle'
 ]

sample_n = min(120000, len(df))
sample_df = df.sample(sample_n, random_state=42).reset_index(drop=True)
X = sample_df[feature_set].copy()
y = sample_df['stage_f_escalation_event'].astype(int).to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print('Stage H sample:', X.shape, 'prevalence:', float(y.mean()))

Stage H sample: (120000, 11) prevalence: 0.011483333333333333


In [3]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            mask = (y_prob >= lo) & (y_prob < hi)
        else:
            mask = (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = float(y_true[mask].mean())
        conf = float(y_prob[mask].mean())
        frac = float(mask.mean())
        ece += abs(acc - conf) * frac
        rows.append({'bin_left': lo, 'bin_right': hi, 'n': int(mask.sum()), 'mean_pred': conf, 'event_rate': acc})
    return float(ece), pd.DataFrame(rows)

baseline_pipe = Pipeline(steps=[
    ('imp', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1200, class_weight='balanced', random_state=42))
])
baseline_pipe.fit(X_train, y_train)
p_test = baseline_pipe.predict_proba(X_test)[:, 1]

auroc = float(roc_auc_score(y_test, p_test))
pr_auc = float(average_precision_score(y_test, p_test))
brier = float(brier_score_loss(y_test, p_test))
ece, cal_df = expected_calibration_error(y_test, p_test, n_bins=12)

perf_df = pd.DataFrame([{
    'model': 'logistic_baseline_stage_h',
    'auroc': auroc,
    'pr_auc': pr_auc,
    'brier': brier,
    'ece': ece,
    'n_test': int(len(y_test))
}])
perf_df.to_csv(TABLE_DIR / 'stage_h_baseline_model_performance.csv', index=False)
cal_df.to_csv(TABLE_DIR / 'stage_h_calibration_curve.csv', index=False)

coef = baseline_pipe.named_steps['clf'].coef_.ravel()
coef_df = pd.DataFrame({'feature': feature_set, 'coef': coef, 'abs_coef': np.abs(coef)}).sort_values('abs_coef', ascending=False)
coef_df.to_csv(TABLE_DIR / 'stage_h_coefficient_summary.csv', index=False)

rng = np.random.default_rng(42)
boot_scores = []
idx = np.arange(len(y_test))
for _ in range(100):
    b = rng.choice(idx, size=len(idx), replace=True)
    yb = y_test[b]
    pb = p_test[b]
    if len(np.unique(yb)) < 2:
        continue
    boot_scores.append({
        'auroc': float(roc_auc_score(yb, pb)),
        'pr_auc': float(average_precision_score(yb, pb)),
        'brier': float(brier_score_loss(yb, pb))
    })
boot_df = pd.DataFrame(boot_scores)
ci_df = pd.DataFrame([{
    'metric': m,
    'mean': float(boot_df[m].mean()),
    'ci_low': float(boot_df[m].quantile(0.025)),
    'ci_high': float(boot_df[m].quantile(0.975)),
    'n_boot': int(len(boot_df))
} for m in ['auroc', 'pr_auc', 'brier']])
ci_df.to_csv(TABLE_DIR / 'stage_h_bootstrap_ci.csv', index=False)

test_df = X_test.copy().reset_index(drop=True)
test_df['y_true'] = y_test
test_df['p_pred'] = p_test
test_df['response_state_active'] = sample_df.loc[X_test.index, 'response_state_active'].to_numpy()
sub_rows = []
for grp, part in test_df.groupby('response_state_active'):
    if part['y_true'].nunique() > 1:
        g_auroc = float(roc_auc_score(part['y_true'], part['p_pred']))
    else:
        g_auroc = np.nan
    sub_rows.append({
        'group': str(grp),
        'n': int(len(part)),
        'auroc': g_auroc,
        'brier': float(brier_score_loss(part['y_true'], part['p_pred']))
    })
sub_df = pd.DataFrame(sub_rows).sort_values('n', ascending=False)
sub_df.to_csv(TABLE_DIR / 'stage_h_subgroup_error_table.csv', index=False)

with open(REPORT_DIR / 'stage_h_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage H Baseline Summary\n')
    f.write(f'auroc: {auroc:.6f}\n')
    f.write(f'pr_auc: {pr_auc:.6f}\n')
    f.write(f'brier: {brier:.6f}\n')
    f.write(f'ece: {ece:.6f}\n')
    f.write(f'n_train: {len(X_train)}\n')
    f.write(f'n_test: {len(X_test)}\n')

print('Stage H baseline artifacts generated')
perf_df

Stage H baseline artifacts generated


,model,auroc,pr_auc,brier,ece,n_test
0,logistic_baseline_stage_h,0.999997,0.999721,0.004913,0.007088,30000


In [4]:
manifest_h = {
    'phase': 'H',
    'notebook': '05_stage_h_coupled_pipeline_design.ipynb',
    'inputs': [
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet'
    ],
    'outputs_tables': [
        'Results/tables/notebook05_stage_h/stage_h_baseline_model_performance.csv',
        'Results/tables/notebook05_stage_h/stage_h_calibration_curve.csv',
        'Results/tables/notebook05_stage_h/stage_h_coefficient_summary.csv',
        'Results/tables/notebook05_stage_h/stage_h_bootstrap_ci.csv',
        'Results/tables/notebook05_stage_h/stage_h_subgroup_error_table.csv'
    ],
    'outputs_reports': [
        'Results/reports/notebook05_stage_h/stage_h_summary.txt'
    ]
}
with open(META_DIR / 'phase_h_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_h, f, indent=4)

proof_h = {
    'baseline_performance_generated': (TABLE_DIR / 'stage_h_baseline_model_performance.csv').exists(),
    'calibration_curve_generated': (TABLE_DIR / 'stage_h_calibration_curve.csv').exists(),
    'bootstrap_ci_generated': (TABLE_DIR / 'stage_h_bootstrap_ci.csv').exists(),
    'subgroup_error_generated': (TABLE_DIR / 'stage_h_subgroup_error_table.csv').exists(),
    'manifest_generated': (META_DIR / 'phase_h_manifest.json').exists()
}
with open(REPORT_DIR / 'stage_h_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof_h}, f, indent=4)

print('Stage H manifest/proof generated')
proof_h

Stage H manifest/proof generated


{'baseline_performance_generated': True,
 'calibration_curve_generated': True,
 'bootstrap_ci_generated': True,
 'subgroup_error_generated': True,
 'manifest_generated': True}

In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook05'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)